# Notebook 02 – Data Cleaning & Stadardization:

## Objective

### Clean every dataset individually before merging.

#### Cleaning includes:-

##### - Standardizing column names
##### - Data types
##### - Missing values
##### - Duplicate removal
##### - Date conversion

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
RAW = Path("data/raw/kshashtra")
PROCESSED = Path("data/processed")

PROCESSED.mkdir(exist_ok=True)

In [3]:
customers = pd.read_csv(RAW/"customers.csv")
orders = pd.read_csv(RAW/"orders.csv")
order_items = pd.read_csv(RAW/"order_line_items.csv")
website_sessions = pd.read_csv(RAW/"website_sessions.csv")
website_daily = pd.read_csv(RAW/"website_daily.csv")
campaigns = pd.read_csv(RAW/"meta_ads_campaigns.csv")
sku_catalog = pd.read_csv(RAW/"sku_catalog.csv")
inventory = pd.read_csv(RAW/"inventory_snapshots.csv")
purchase_orders = pd.read_csv(RAW/"purchase_orders.csv")

In [4]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "website_sessions": website_sessions,
    "website_daily": website_daily,
    "campaigns": campaigns,
    "sku_catalog": sku_catalog,
    "inventory": inventory,
    "purchase_orders": purchase_orders
}

### Standardize column names:

In [5]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

In [6]:
for name in datasets:
    datasets[name] = clean_columns(datasets[name])
print("Column names standardized..!")

Column names standardized..!


### Checking Duplicates:

In [7]:
duplicate_summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Duplicate Rows": [
        df.duplicated().sum()
        for df in datasets.values()
    ]
})
duplicate_summary

,Dataset,Duplicate Rows
0,customers,0
1,orders,0
2,order_items,13
3,website_sessions,2
4,website_daily,0
5,campaigns,0
6,sku_catalog,0
7,inventory,0
8,purchase_orders,0


In [8]:
for name in datasets:
    datasets[name] = datasets[name].drop_duplicates() 

### Checking for missing values:

missing = pd.DataFrame({
    name: df.isnull().sum()
    for name, df in datasets.items()
}).fillna(0)
missing

### Converting Date Columns:

In [9]:
for name, df in datasets.items():
    for col in df.columns:
        if "date" in col.lower():
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

### New Dimensions:

In [10]:
summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})
summary

,Dataset,Rows,Columns
0,customers,16865,11
1,orders,30000,16
2,order_items,42108,10
3,website_sessions,475656,14
4,website_daily,21915,15
5,campaigns,50,20
6,sku_catalog,55,5
7,inventory,11495,8
8,purchase_orders,1801,8


In [11]:
for name, df in datasets.items():
    df.to_csv(
        PROCESSED/f"{name}.csv",
        index=False
    )
print("Cleaned datasets saved..!")

Cleaned datasets saved..!


## Summary

#### - Column standardized
#### - Duplicate removed
#### - Missing value inspected
#### - Date conversion done
#### - Saved cleaned datasets

In [12]:
sku_catalog.rename(
    columns={"sku": "sku_id"},
    inplace=True
)